# 03 — Exploratory analysis

Phase 3, deliverable 2. What is in this dataset, at what scale, and with what shape.

**Rule for every number in this notebook and the next two:** it comes from the
database or from `data/truth/_truth.json`. Never from spec prose — the prose in
`docs/01_phase2_data_architecture.md` describes an earlier parameterisation and is
superseded (limitation L8).

Nothing here touches schema `truth` except to read the published truth file. The
SQL library runs as the restricted `analyst` role, which is denied on `truth`;
`scripts/05_crosscheck.py` asserts that denial as part of the metric check.


In [ ]:
import sys, json; from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.analysis import funnel as F

TABLES = F.load_tables()
TRUTH  = json.loads((ROOT / 'data/truth/_truth.json').read_text())
pd.set_option('display.width', 160)
print({k: len(v) for k, v in TABLES.items()})


## Scale and grain


In [ ]:
pd.DataFrame([
    {'table': k, 'rows': len(v), 'columns': v.shape[1]}
    for k, v in sorted(TABLES.items())
])


## Order value is a right-skewed mixture, not a point mass

Phase 1 pins the **mean** GMV at ₹1,000; the median sits far below it. A dataset
where every order is ₹1,000 would make every value-band analysis vacuous, so the
skew is deliberate (decision A34).


In [ ]:
o = TABLES['fct_order']
o[['gmv', 'order_value', 'discount_pct', 'quantity']].describe().T


In [ ]:
q = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
pd.DataFrame({'gmv_quantile': o['gmv'].quantile(q)})


## The planted truth, for reference

Everything downstream is checked against these. They are *measured* post-generation,
not asserted before it.


In [ ]:
effects = TRUTH['planted_causal_effects']['cod_on_rto']
pd.Series({
    'planted is_cod coefficient': effects['logit_coefficient'],
    'true AME (pp)':              round(effects['average_marginal_effect_pp'], 2),
    'naive observed gap (pp)':    round(effects['naive_observed_gap_pp'], 2),
    'selection share of gap':     round(effects['selection_share_of_naive_gap'], 3),
    'naive / truth':              round(effects['naive_over_truth_multiple'], 2),
    'AUC ceiling':                round(TRUTH['achieved']['auc_ceiling_precheckout'], 4),
})


> The naive comparison reports **1.77x the truth**. Recovering how much of that
> gap adjustment can close — and demonstrating that it *cannot* close all of it —
> is notebook 05's job and the point of the whole project.
